
# Step 02: Extract first and last corresponding author information

Input:
    data/interim/publication/1_AllPublications.csv

Output:
    data/interim/publication/2_first_bycountry
    data/interim/publication/2_last_bycountry

    data/interim/publication/"2_first_NameAndCountry.csv"
    data/interim/publication/"2_last_NameAndCountry.csv"

Purpose:
    - Extract authors name
    - Identify address and affiliated countries

In [ ]:
import pandas as pd
from tqdm import tqdm
from rapidfuzz import fuzz
import re
import string
import numpy as np

from pathlib import Path

from geotext import GeoText
import pycountry
import os


In [ ]:

# -----------------------------
# Paths
# -----------------------------
PROJECT_ROOT = Path(__file__).resolve().parents[2]
IN_FILE = PROJECT_ROOT / "data" / "interim" / "publication" / "1_AllPublications.csv"

OUT_DIR_FIRST = PROJECT_ROOT / "data" / "interim" / "publication"/ "2_first_bycountry"
OUT_DIR_FIRST.mkdir(parents=True, exist_ok=True)

OUT_DIR_LAST = PROJECT_ROOT / "data" / "interim" / "publication"/ "2_last_bycountry"
OUT_DIR_LAST.mkdir(parents=True, exist_ok=True)

OUT_FIRST = PROJECT_ROOT / "data" / "interim" / "publication"/ "2_first_NameAndCountry.csv"
OUT_LAST = PROJECT_ROOT / "data" / "interim" / "publication"/ "2_last_NameAndCountry.csv"

# 1 First author

In [ ]:
# Path to your CSV file
cols_to_import = ['index_num', 'Addresses', 'Affiliations','Author Full Names']

df = pd.read_csv(IN_FILE, usecols=cols_to_import)
df

In [ ]:
def Constru_Author_info(df):
    fst_authors_info_lst=[]
    for index, row in tqdm(df.iterrows(), total=df.shape[0]): # process row by row
        if isinstance(row['Author Full Names'], str) and isinstance(row['Addresses'], str):
            address = "" # create empty address in case not found
            country = "" # create empty country in case not found
            authors = row['Author Full Names'].split(';') # extract author name
            authors_num = len(authors)
            fst_author=authors[0]
            temp = re.split(r'[\[\]]',row['Addresses'])
            for ind, string in enumerate(temp):
                if fst_author in string:
                    if ind+1<=len(temp)-1:
                        address = temp[ind+1] # find author name in address, and identify address accordingly
                elif ind == len(temp)-1 and address == "":
                    address = row['Addresses'] # in case author name and address cannot be splitted
            if address:
                temp1 = re.split(r'[ ;&,]',address) # if address is not empty, find the last word as country
                temp1=[x for x in temp1 if x !='']
                country = temp1[-1]
                
            fst_authors_info_lst.append([fst_author, authors_num, address, country, row['Affiliations'], row['index_num']]) # combine info found above
            
    fst_authors_info_df = pd.DataFrame(fst_authors_info_lst, columns=["author", "number of authors", "address", "country1", "affiliations", "index_num"])
    fst_authors_info_df = fst_authors_info_df.replace("",np.nan)
    
    return fst_authors_info_df

In [ ]:
fst_authors_info_df = Constru_Author_info(df)
fst_authors_info_df

In [ ]:
# Deal with empyt address

na_address=fst_authors_info_df.loc[fst_authors_info_df['address'].isna(),'address']
short_address=fst_authors_info_df.loc[fst_authors_info_df['address'].str.len()<15,'address']

outcome = fst_authors_info_df.sort_values(by='index_num')
short_address=outcome.loc[outcome['address'].isna(),'address']

print(short_address)

In [ ]:
# Unify country names

tqdm.pandas()
outcome = outcome.replace(np.nan,"")

def extract_country(address):
    places = GeoText(address)
    return places.countries[0] if places.countries else None

def get_country_name(address):
    for country in pycountry.countries:
        if country.name in address:
            return country.name
    return None

outcome['country2'] = outcome['address'].progress_apply(extract_country)

# Fill missing country2 with country1
outcome['country2'] = outcome['country2'].fillna(outcome['country1'])

# Upper case all country names
outcome['country2'] = outcome['country2'].str.upper()

# replace 'HONG KONG'and'TAIWAN' with 'CHINA'
outcome['country2'] = outcome['country2'].replace(['HONG KONG', 'TAIWAN'], 'CHINA')
outcome

In [ ]:
# Check country names
print(outcome['country2'].unique())

# mapping corrections
country_fix_map = {
    "USA": "UNITED STATES",
    "UNITED STATES": "UNITED STATES",
    "UNIT": "UNITED KINGDOM",   
    "ENGLAND": "UNITED KINGDOM",
    "SCOTLAND": "UNITED KINGDOM",
    "WALES": "UNITED KINGDOM",
    "SCOTLAND?": "UNITED KINGDOM",
    "UNITED KINGDOM": "UNITED KINGDOM",
    "NORTH KOREA": "KOREA, DEMOCRATIC PEOPLE'S REPUBLIC OF",
    "SOUTH KOREA": "KOREA, REPUBLIC OF",
    "TURKIYE": "TURKEY",
    "MACEDONIA": "NORTH MACEDONIA",
    "YUGOSLAVIA": "SERBIA",  # or SERBIA & MONTENEGRO historically
    "HERCEG": "BOSNIA AND HERZEGOVINA",
    "MONTENEG": "MONTENEGRO",
    "IVOIRE": "CÔTE D'IVOIRE",
    "IVORY COAST": "CÔTE D'IVOIRE",
    "ISLAND": "ICELAND",
    "REP": "REPUBLIC OF KOREA",  # ambiguous → could be Korea
    "REPUBL": "REPUBLIC OF KOREA",  # same
    "BISSAU": "GUINEA-BISSAU",
    "LUCIA": "SAINT LUCIA",
    "ANTILLES": "NETHERLANDS ANTILLES",  # historical
    "ST": "SAINT KITTS AND NEVIS",
    "NEVI": "SAINT KITTS AND NEVIS",
    "TOBAGO": "TRINIDAD AND TOBAGO",
    "CAICOS": "TURKS AND CAICOS ISLANDS",
    "CURACAO": "CURAÇAO",
    "PRIN": "SAO TOME AND PRINCIPE",
    "MONTSERRAT": "MONTSERRAT",
    "LYKKEB&UAELIG": np.nan,   # garbage
    "471003": np.nan,          # postal code
    "2205": np.nan,
    "*": np.nan,
    
    "REP": np.nan,        # "REP" → Czech Republic
    "REPUBL": np.nan, # "REPUBL" fragment
    "HERCEG": "BOSNIA AND HERZEGOVINA",
    "ISLAND": np.nan,            # ambiguous, but most likely Iceland
    "NEVI": "SAINT KITTS AND NEVIS",
    "BARBU": "ANTIGUA AND BARBUDA",
    "LUCIA": "SAINT LUCIA",
    "VINCENT": "SAINT VINCENT AND THE GRENADINES",
    "BISSAU": "GUINEA-BISSAU",
    "CAICOS": "TURKS AND CAICOS ISLANDS",
    "HELENA": "SAINT HELENA",
    "S-75": np.nan,                     # not a country → remove
    "MORINOSATOP": np.nan,              # unknown → remove
    "": np.nan,

    # United States
    "USA": "UNITED STATES",
    "UNITED STATES": "UNITED STATES",
    "AMERICA": "UNITED STATES",

    # United Kingdom & parts
    "ENGLAND": "UNITED KINGDOM",
    "SCOTLAND": "UNITED KINGDOM",
    "WALES": "UNITED KINGDOM",
    "SCOTLAND?": "UNITED KINGDOM",
    "UNITED KINGDOM": "UNITED KINGDOM",
    "GIBRALTAR": "UNITED KINGDOM",
    "ISLAND": np.nan,   # likely truncation
    "HELENA": "SAINT HELENA, ASCENSION AND TRISTAN DA CUNHA",
    "CAICOS": "UNITED KINGDOM",   # fragment of Turks & Caicos
    "BRITISH VIRGIN ISLANDS": "UNITED KINGDOM",
    "?ENGLAND":"UNITED KINGDOM",
    
    # Korea
    "SOUTH KOREA": "KOREA, REPUBLIC OF",
    "NORTH KOREA": "KOREA, DEMOCRATIC PEOPLE'S REPUBLIC OF",

    # Ivory Coast
    "IVORY COAST": "CÔTE D'IVOIRE",
    "IVOIRE": "CÔTE D'IVOIRE",

    # Eswatini
    "SWAZILAND": "ESWATINI",

    # Myanmar
    "BURMA": "MYANMAR",

    # North Macedonia
    "MACEDONIA": "NORTH MACEDONIA",
    
    "TURKIYE": "TURKEY",

    # Caribbean & overseas territories
    "ANTILLES": "FRANCE", # French Antilles
    "BONAIRE": "NETHERLANDS",
    "BES ISLANDS": "NETHERLANDS",  # Bonaire, Sint Eustatius, Saba

    # Small islands (independent)
    "SAMOA": "SAMOA",
    "AMERICAN SAMOA": "UNITED STATES",
    "TONGA": "TONGA",
    "FIJI": "FIJI",
    "PALAU": "PALAU",
    "VANUATU": "VANUATU",
    "SOLOMON ISLANDS": "SOLOMON ISLANDS",
    "KIRIBATI": "KIRIBATI",
    "NAURU": "NAURU",
    "MARSHALL ISLANDS": "MARSHALL ISLANDS",

    # Others obvious fixes
    "NEVI": "SAINT KITTS AND NEVIS",
    "LUCIA": "SAINT LUCIA",
    "VINCENT": "SAINT VINCENT AND THE GRENADINES",
    "BARBU": "ANTIGUA AND BARBUDA",
    "PRIN": "SAO TOME AND PRINCIPE",
    "BISSAU": "GUINEA-BISSAU",
    "SAUDI ARABIA": "SAUDI ARABIA",
    "UAE": "UNITED ARAB EMIRATES",
    "EMIRATES": "UNITED ARAB EMIRATES",

    # Likely noise/non-countries → drop or map to None
    "S-75": np.nan,
    "A.": np.nan,
    "B.": np.nan,
    "C.": np.nan,
    "E.": np.nan,
    "F.": np.nan,
    "G.": np.nan,
    "J.": np.nan,
    "K.": np.nan,
    "L.": np.nan,
    "M.": np.nan,
    "P.": np.nan,
    "S.": np.nan,
    "T.": np.nan,
    "Y.": np.nan,
    "R.": np.nan,
    "REP": np.nan,       # truncated
    "REPUBL": np.nan,    # truncated
    "DISORDER": np.nan,
    "MORINOSATOP": np.nan,
    "NEUROSCI": np.nan,
    "ALEXANDRA": np.nan,
    "PARC": np.nan,
    "TICSE": np.nan,
    "61005":np.nan,
    "SCI":np.nan,
    "H-4032":np.nan,
    "I-00185":np.nan,
}

# apply mapping
outcome['country_clean'] = (
    outcome['country2']
    .replace(country_fix_map)
    .str.strip()
    .str.upper()
)

# check remaining unique values
print(sorted(outcome['country_clean'].dropna().unique()))

In [ ]:
# Summarize country counts and proportions
country_counts = outcome['country_clean'].value_counts()


country_proportions = country_counts / country_counts.sum()

result = pd.DataFrame({
    'Count': country_counts,
    'Proportion': country_proportions
})
result

In [ ]:
# Path to your CSV file
outcome = outcome.rename(columns={'country_clean': 'country'})

outcome.to_csv(OUT_FIRST, index=False)
print(f"Extracted author info saved to {OUT_FIRST}")

In [ ]:
# Export to separate files for gender identification
outcome = outcome[['author', 'index_num', 'country']]

def clean_filename(filename):
    valid_chars = "-_.() %s%s" % (string.ascii_letters, string.digits)
    cleaned_filename = ''.join(c for c in filename if c in valid_chars)
    return cleaned_filename


# 根据country进行分组
grouped = outcome.groupby('country')

# 遍历每个分组，并保存到CSV文件
output_path = OUT_DIR_FIRST
for name, group in grouped:
    # 清洗国家名以避免文件保存错误
    cleaned_name = clean_filename(name)
    filename = f'code_{cleaned_name}.csv'
    group.to_csv(os.path.join(output_path, filename), index=False)

# 2 Last authors

In [ ]:
def match_corresponding_authors(row, full_names):
    temps = re.split(r'\(通讯作者\)|\(corresponding author\)', row['Reprint Addresses'])
    full_names_matched = []
    #制作一个列表用来存放同一行通讯作者的完整名称、国家、性别数据，三个一组
    author_info = []
    result_list = []  # 提前初始化result_list以确保它总是有一个值

    if len(temps) == 2:
        names = temps[0].split(';')#对同一地址的多位通讯作者名字进行拆分
        parts = temps[1]#存放通讯作者地址信息
        for i,name in enumerate(names, start=1):
            best_match = max(full_names, key=lambda full_name: fuzz.ratio(name, full_name))
            #full_names_matched.append(best_match)
            
            if parts:
                author_info.append(best_match)
                author_info.append(parts)  # 将parts[-1]作为一个元素添加到author_info列表中
            else:
                author_info.extend(best_match)
                author_info.append("NULL")  # 如果parts为空，则添加"NULL"作为元素

            
    elif len(temps) > 2:
        for index,temp in enumerate(temps):
            if index == 0 :
                names = temp.split(';')
                parts = temps[index + 1].split(';')[0]
                for i,name in enumerate(names, start=1):
                    best_match = max(full_names, key=lambda full_name: fuzz.ratio(name, full_name))
                    author_info.append(best_match)
                    author_info.append(parts)

                        
            else :
                if index < len(temps) - 1:
                    names = temp.split(';')[1:]
                    parts = temps[index + 1].split(';')[0]
                    for i,name in enumerate(names, start=1):
                        best_match = max(full_names, key=lambda full_name: fuzz.ratio(name, full_name))
                        author_info.append(best_match)
                        author_info.append(parts)

            

    # 对作者信息去重，只根据每组的第一个元素来判断是否去重
    seen_first_elements = set()
    for i in range(0, len(author_info), 2):  # 假设author_info的长度总是偶数
        first_element = author_info[i]
        if first_element not in seen_first_elements:
            # 如果第一个元素未出现过，添加当前元素及其配对的元素到结果列表
            result_list.extend(author_info[i:i+2])
            seen_first_elements.add(first_element)
    return result_list

#def clean_author(author):
    # 使用正则表达式去除开头和结尾的空格和标点符号
#    cleaned_author = re.sub(r'^[\s' + re.escape(string.punctuation) + ']+|[\s' + re.escape(string.punctuation) + ']+$', '', author)
#    return cleaned_author

def clean_author(author):
    pattern = rf'^[\s{re.escape(string.punctuation)}]+|[\s{re.escape(string.punctuation)}]+$'
    return re.sub(pattern, '', author)


def find_full_corresponding_authors_pandas(df):
    # Initialize new columns
    #df['corresponding author full'] = ''
    df['test_info'] = ''
    unique_authors = []  # 用于存储不重复的作者信息
    for i in range(1, 8):  # 创建每个人的姓名和地址列表
        df[f'author{i}'] = ''
        df[f'address{i}'] = ''
    
    # Use tqdm to show progress
    for index, row in tqdm(df.iterrows(), total=df.shape[0]):
        if isinstance(row['Author Full Names'], str) and isinstance(row['Reprint Addresses'], str):
            full_names = row['Author Full Names'].split(';')
            author_information = match_corresponding_authors(row, full_names)
            # 使用列表推导式将每个元素转换为字符串
            author_information_str = '; '.join([str(item) for item in author_information])
            # 将转换后的字符串赋值给DataFrame的 'test_info' 单元格
            df.at[index, 'test_info'] = author_information_str
            #index编号
            index_num = row['index_num']
            
            for i in range(0, min(len(author_information), 14), 2):
                author_col_name = f'author{(i//2)+1}'
                address_col_name = f'address{(i//2)+1}'
                author = clean_author(author_information[i])
                address = author_information[i + 1] if i + 1 < len(author_information) else 'Unknown'
                df.at[index, author_col_name] = author
                df.at[index, address_col_name] = address

                unique_authors.append([author, address, index_num])

    cleaned_authors = set((clean_author(author), address, index_num) for author, address, index_num in unique_authors)
    unique_authors = [list(author_address) for author_address in cleaned_authors]
    
    unique_authors_df = pd.DataFrame(unique_authors, columns=["author", "address", "index_num"])
    # 提取国家信息并转换为大写形式
    unique_authors_df['country'] = unique_authors_df['address'].apply(lambda x: x.split(' ')[-1].strip().upper().rstrip('.'))
    return df, unique_authors_df

In [ ]:
unique_authors = find_full_corresponding_authors_pandas(df)
    
    # Write the updated DataFrame to a new CSV
unique_authors.to_csv(OUT_LAST, index=False)
print(f"Updated CSV saved to {OUT_LAST}")

unique_authors.sort_values(by='index_num') 

outcome = unique_authors

In [ ]:
# Identify country for last authors

tqdm.pandas()

def extract_country(address):
    places = GeoText(address)
    return places.countries[0] if places.countries else None

def get_country_name(address):
    for country in pycountry.countries:
        if country.name in address:
            return country.name
    return None

outcome['country2'] = outcome['address'].progress_apply(extract_country)
outcome['country2'] = outcome['country2'].fillna(outcome['country'])

# Transform to upper case
outcome['country2'] = outcome['country2'].str.upper()

# Replace'HONG KONG'and 'TAIWAN'with 'CHINA'
outcome['country2'] = outcome['country2'].replace(['HONG KONG', 'TAIWAN'], 'CHINA')
outcome

print(outcome['country2'].unique())

In [ ]:
# Update country names
# mapping corrections
country_fix_map = {
    "USA": "UNITED STATES",
    "UNITED STATES": "UNITED STATES",
    "UNIT": "UNITED KINGDOM",   # likely truncated
    "ENGLAND": "UNITED KINGDOM",
    "SCOTLAND": "UNITED KINGDOM",
    "WALES": "UNITED KINGDOM",
    "?ENGLAND": "UNITED KINGDOM",
    "SCOTLAND?": "UNITED KINGDOM",
    "UNITED KINGDOM": "UNITED KINGDOM",
    "NORTH KOREA": "KOREA, DEMOCRATIC PEOPLE'S REPUBLIC OF",
    "SOUTH KOREA": "KOREA, REPUBLIC OF",
    "TURKIYE": "TURKEY",
    "MACEDONIA": "NORTH MACEDONIA",
    "YUGOSLAVIA": "SERBIA",  # or SERBIA & MONTENEGRO historically
    "HERCEG": "BOSNIA AND HERZEGOVINA",
    "MONTENEG": "MONTENEGRO",
    "IVOIRE": "CÔTE D'IVOIRE",
    "IVORY COAST": "CÔTE D'IVOIRE",
    "ISLAND": "ICELAND",
    "REP": "REPUBLIC OF KOREA",  # ambiguous → could be Korea
    "REPUBL": "REPUBLIC OF KOREA",  # same
    "BISSAU": "GUINEA-BISSAU",
    "LUCIA": "SAINT LUCIA",
    "ANTILLES": "NETHERLANDS ANTILLES",  # historical
    "ST": "SAINT KITTS AND NEVIS",
    "NEVI": "SAINT KITTS AND NEVIS",
    "TOBAGO": "TRINIDAD AND TOBAGO",
    "CAICOS": "TURKS AND CAICOS ISLANDS",
    "CURACAO": "CURAÇAO",
    "PRIN": "SAO TOME AND PRINCIPE",
    "MONTSERRAT": "MONTSERRAT",
    "LYKKEB&UAELIG": np.nan,   # garbage
    "471003": np.nan,          # postal code
    "2205": np.nan,
    "*": np.nan
}

# apply mapping
outcome['country_clean'] = (
    outcome['country2']
    .replace(country_fix_map)
    .str.strip()
    .str.upper()
)

# check remaining unique values
print(sorted(outcome['country_clean'].dropna().unique()))

In [ ]:
# count and proportion

country_counts = outcome['country_clean'].value_counts()


country_proportions = country_counts / country_counts.sum()


result = pd.DataFrame({
    'Count': country_counts,
    'Proportion': country_proportions
})

result

In [ ]:
# Export to separate files for gender identification
outcome = outcome[['author', 'index_num', 'country_clean']]
outcome = outcome.rename(columns={'country_clean': 'country'})

def clean_filename(filename):
    valid_chars = "-_.() %s%s" % (string.ascii_letters, string.digits)
    cleaned_filename = ''.join(c for c in filename if c in valid_chars)
    return cleaned_filename

In [ ]:

grouped = outcome.groupby('country')

output_path = OUT_DIR_LAST
for name, group in grouped:
    cleaned_name = clean_filename(name)
    filename = f'code_{cleaned_name}.csv'
    group.to_csv(os.path.join(output_path, filename), index=False)